In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from lightgbm import LGBMClassifier
from sklearn.cluster import KMeans
sns.set_theme()

In [ ]:
df_map = pd.read_csv(
    r'C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\acidentes_pbic_somas.csv',
    encoding='utf-8',
    parse_dates=['data_inversa'],
    dayfirst=True,
    low_memory=False
)

print("Shape inicial:", df_map.shape)


In [ ]:
df_map = df_map.copy()

df_map = df_map.drop_duplicates(subset=["id", "pesid"])

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
from datetime import datetime

# =============================
# Limpeza da BR
# =============================

def limpar_br(df):
    return (
        df["br"]
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.replace("BR-", "", regex=False)
        .str.strip()
    )

df_map["br"] = limpar_br(df_map)

# =============================
# FILTRO DE RODOVIA
# =============================

br_escolhida = input("Digite a BR que deseja visualizar (ex: 116) ou pressione ENTER para todas: ")

df_filtrado = df_map.copy()

if br_escolhida.strip() != "":
    df_filtrado = df_filtrado[df_filtrado["br"] == br_escolhida.strip()]
    print(f"Filtrando apenas BR {br_escolhida}")
else:
    print("Mostrando todas as rodovias")

if df_filtrado.empty:
    print("Nenhum dado encontrado para essa BR.")
    exit()

# =============================
# Limpeza coordenadas
# =============================

cols = ["latitude", "longitude", "km"]

for col in cols:
    df_filtrado[col] = pd.to_numeric(
        df_filtrado[col].astype(str).str.replace(",", "."),
        errors="coerce"
    )

df_filtrado = df_filtrado.dropna(subset=["latitude", "longitude", "km"])

# =============================
# KM BIN
# =============================

df_filtrado["km_bin"] = ((df_filtrado["km"] // 5) * 5).astype(int)

# =============================
# Índice de severidade (AGORA FILTRADO)
# =============================

df_filtrado["severity_index"] = (
    4 * df_filtrado["Sum_Mortos"] +
    2 * df_filtrado["Sum_Feridos_Graves"] +
    1 * df_filtrado["Sum_Feridos_Leves"]
)

# =============================
# Agrupamento (AGORA FILTRADO)
# =============================

df_segmento = df_filtrado.groupby(["br", "uf", "km_bin"]).agg(
    severity_index=("severity_index", "mean"),
    n_acidentes=("severity_index", "count"),
    total_mortos=("Sum_Mortos", "sum"),
    total_feridos_graves=("Sum_Feridos_Graves", "sum"),
    total_feridos_leves=("Sum_Feridos_Leves", "sum")
).reset_index()
# =============================
# Classificação de veículos
# =============================

caminhao = ["Caminhão", "Caminhão-trator"]
carros_leves = ["Automóvel"]
caminhonetes = ["Caminhonete", "Camioneta", "Utilitário"]
motos = ["Motocicleta", "Motoneta"]
ciclomotores = ["Ciclomotor", "Triciclo", "Quadriciclo"]

# NOVO 🚍
onibus = ["Ônibus", "Micro-ônibus"]

# =============================
# Criar mapa
# =============================

m = folium.Map(
    location=[df_filtrado["latitude"].median(), df_filtrado["longitude"].median()],
    zoom_start=6
)

# =============================
# Camadas
# =============================

layer_todos = MarkerCluster(name="🔴 Todos acidentes com mortos").add_to(m)
layer_caminhao = MarkerCluster(name="🚛 Caminhões").add_to(m)
layer_carro = MarkerCluster(name="🚗 Automóveis").add_to(m)
layer_caminhonete = MarkerCluster(name="🚙 Caminhonetes").add_to(m)
layer_moto = MarkerCluster(name="🏍 Motocicletas").add_to(m)
layer_ciclomotor = MarkerCluster(name="🛵 Ciclomotores").add_to(m)

# NOVO 🚍
layer_onibus = MarkerCluster(name="🚌 Ônibus").add_to(m)

# =============================
# Agrupamento auxiliar
# =============================

segmentos_grupo = df_filtrado.groupby(["br", "uf", "km_bin"])

# =============================
# Plotagem
# =============================

for _, row in df_segmento.iterrows():

    try:
        grupo = segmentos_grupo.get_group(
            (row["br"], row["uf"], row["km_bin"])
        )
    except KeyError:
        continue

    lat = grupo["latitude"].median()
    lon = grupo["longitude"].median()

    if np.isnan(lat) or np.isnan(lon):
        continue

    # Flags de veículos
    tem_caminhao = grupo["tipo_veiculo"].isin(caminhao).any()
    tem_carro = grupo["tipo_veiculo"].isin(carros_leves).any()
    tem_caminhonete = grupo["tipo_veiculo"].isin(caminhonetes).any()
    tem_moto = grupo["tipo_veiculo"].isin(motos).any()
    tem_ciclomotor = grupo["tipo_veiculo"].isin(ciclomotores).any()
    tem_onibus = grupo["tipo_veiculo"].isin(onibus).any()  # NOVO

    # Popup
    veiculos = grupo["tipo_veiculo"].value_counts().head(3)
    veiculo_txt = "<br>".join([f"{v}: {c}" for v, c in veiculos.items()])

    popup_html = f"""
    <b>BR:</b> {row['br']} - {row['uf']} | <b>KM:</b> {row['km_bin']}<br>
    <b>Total de acidentes:</b> {row['n_acidentes']}<br>
    <b>Total de mortos:</b> {row['total_mortos']}<br>
    <hr><b>Top Veículos:</b><br>{veiculo_txt}
    """

    radius = 7 + (np.log1p(row["total_mortos"]) * 3)

    # Geral
    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        color="red",
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(layer_todos)

    # Camadas específicas
    if tem_caminhao:
        folium.CircleMarker([lat, lon], radius=radius, color="purple", fill=True, fill_opacity=0.7, popup=popup_html).add_to(layer_caminhao)

    if tem_carro:
        folium.CircleMarker([lat, lon], radius=radius, color="blue", fill=True, fill_opacity=0.7, popup=popup_html).add_to(layer_carro)

    if tem_caminhonete:
        folium.CircleMarker([lat, lon], radius=radius, color="darkgreen", fill=True, fill_opacity=0.7, popup=popup_html).add_to(layer_caminhonete)

    if tem_moto:
        folium.CircleMarker([lat, lon], radius=radius, color="orange", fill=True, fill_opacity=0.7, popup=popup_html).add_to(layer_moto)

    if tem_ciclomotor:
        folium.CircleMarker([lat, lon], radius=radius, color="black", fill=True, fill_opacity=0.7, popup=popup_html).add_to(layer_ciclomotor)

    # NOVO 🚍
    if tem_onibus:
        folium.CircleMarker(
            [lat, lon],
            radius=radius,
            color="pink",
            fill=True,
            fill_opacity=0.7,
            popup=popup_html
        ).add_to(layer_onibus)

# =============================
# Controle
# =============================

folium.LayerControl(collapsed=False).add_to(m)

# =============================
# Salvar
# =============================

path = f"C:/Fause/Programas/EngSoft/repos/ML-Transportes/PBIC\EtapaML\Final\MapsCarro/mapa_mortos_carros.html"

m.save(path)

print(f"Mapa gerado com sucesso em: {path}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# =============================
# 1️⃣ Definir veículos de caminhão
# =============================

caminhao = ["Caminhão", "Caminhão-trator"]

# =============================
# 2️⃣ Filtrar base original (IMPORTANTE)
# =============================

df_caminhao = df_filtrado[
    df_filtrado["tipo_veiculo"].isin(caminhao)
].copy()

# Segurança: remover linhas inválidas
df_caminhao = df_caminhao.dropna(subset=["br", "uf", "km_bin"])

# =============================
# 3️⃣ Criar índice de severidade
# =============================

df_caminhao["severity_index"] = (
    4 * df_caminhao["Sum_Mortos"] +
    2 * df_caminhao["Sum_Feridos_Graves"] +
    1 * df_caminhao["Sum_Feridos_Leves"]
)

# =============================
# 4️⃣ Agrupar por segmento
# =============================

df_segmento_caminhao = df_caminhao.groupby(
    ["br", "uf", "km_bin"]
).agg(
    n_acidentes=("Sum_Total_Vitimas", "count"),
    total_mortos=("Sum_Mortos", "sum"),
    total_feridos_graves=("Sum_Feridos_Graves", "sum"),
    total_feridos_leves=("Sum_Feridos_Leves", "sum")
).reset_index()

# =============================
# 5️⃣ Criar nome do segmento
# =============================

df_segmento_caminhao["segmento"] = (
    "BR-" + df_segmento_caminhao["br"].astype(str) + " (" +
    df_segmento_caminhao["uf"] + ") - km " +
    df_segmento_caminhao["km_bin"].astype(str)
)

# =============================
# 7️⃣ TOP 10 - IMPACTO GRAVE
# =============================

df_segmento_caminhao["impacto_grave"] = (
    df_segmento_caminhao["total_mortos"] +
    df_segmento_caminhao["total_feridos_graves"]
)

top10_impacto_caminhao = df_segmento_caminhao.sort_values(
    "impacto_grave", ascending=False
).head(10)

plt.figure()

plt.barh(
    top10_impacto_caminhao["segmento"],
    top10_impacto_caminhao["impacto_grave"]
)

plt.xlabel("Mortos + Feridos Graves (Caminhões)")
plt.title("Top 10 segmentos com maior impacto grave (Caminhões)")
plt.gca().invert_yaxis()

plt.show()



In [ ]:
def consulta_por_variavel(df_original, df_top10, coluna_consulta):
    """
    df_original: df_caminhao (ou df_filtrado já tratado)
    df_top10: top10_impacto_caminhao
    coluna_consulta: string (ex: 'Fabricante', 'tipo_acidente')
    """

    # =============================
    # 1️⃣ Filtrar apenas segmentos do Top 10
    # =============================
    
    df_filtrado_top = df_original.merge(
        df_top10[["br", "uf", "km_bin"]],
        on=["br", "uf", "km_bin"],
        how="inner"
    )

    # =============================
    # 2️⃣ Agrupar dinamicamente
    # =============================
    
    resultado = df_filtrado_top.groupby(
        ["br", "uf", "km_bin", coluna_consulta]
    ).size().reset_index(name="quantidade")

    # =============================
    # 3️⃣ Criar nome do segmento
    # =============================
    
    resultado["segmento"] = (
        "BR-" + resultado["br"].astype(str) + " (" +
        resultado["uf"] + ") - km " +
        resultado["km_bin"].astype(str)
    )

    # =============================
    # 4️⃣ Ordenar resultados
    # =============================
    
    resultado = resultado.sort_values(
        ["segmento", "quantidade"],
        ascending=[True, False]
    )

    return resultado

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "Fabricante"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["Fabricante", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "tipo_veiculo"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["tipo_veiculo", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "causa_acidente"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["causa_acidente", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "tipo_acidente"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["tipo_acidente", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "fase_dia"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["fase_dia", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "condicao_metereologica"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["condicao_metereologica", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "tipo_pista"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["tipo_pista", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "ano_fabricacao_veiculo"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["ano_fabricacao_veiculo", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "Modelo_Grupo"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["Modelo_Grupo", "quantidade"]].to_string(index=False))

In [ ]:
res_fabricante = consulta_por_variavel(
    df_caminhao,
    top10_impacto_caminhao,
    "Modelo"
)

for segmento, grupo in res_fabricante.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(grupo[["Modelo", "quantidade"]].to_string(index=False))

In [ ]:
def consulta_multiplas_variaveis(df_original, df_top10, colunas):
    """
    colunas: lista de colunas
    ex: ['Fabricante', 'Modelo_Grupo', 'Modelo', 'ano_fabricacao_veiculo']
    """

    # =============================
    # 1️⃣ Filtrar Top 10
    # =============================
    
    df_filtrado_top = df_original.merge(
        df_top10[["br", "uf", "km_bin"]],
        on=["br", "uf", "km_bin"],
        how="inner"
    )

    # =============================
    # 2️⃣ Agrupar por múltiplas colunas
    # =============================
    
    resultado = df_filtrado_top.groupby(
        ["br", "uf", "km_bin"] + colunas
    ).size().reset_index(name="quantidade")

    # =============================
    # 3️⃣ Criar segmento
    # =============================
    
    resultado["segmento"] = (
        "BR-" + resultado["br"].astype(str) + " (" +
        resultado["uf"] + ") - km " +
        resultado["km_bin"].astype(str)
    )

    # =============================
    # 4️⃣ Ordenar
    # =============================
    
    resultado = resultado.sort_values(
        ["segmento", "quantidade"],
        ascending=[True, False]
    )

    return resultado

In [ ]:
colunas = [
    "Fabricante",
    "Modelo_Grupo",
    "Modelo",
    "ano_fabricacao_veiculo"
]


res_veiculos = consulta_multiplas_variaveis(
    df_caminhao,
    top10_impacto_caminhao,
    colunas
)

In [ ]:
for segmento, grupo in res_veiculos.groupby("segmento"):
    print(f"\n===== {segmento} =====")
    print(
        grupo[colunas + ["quantidade"]]
        .head(10)  # top 5
        .to_string(index=False)
    )